# M2L2 - Introduction To Sequencing Techniques and Data Structures

## Sequencing for RNA-Seq
The generation of sequences from a sample set and the basics of library assembly will be discussed. In this notebook, we will: 
- Explore how Illumina Sequencing functions and understand how Libraries are prepared
- Discuss the impacts of sequencing choices (paired-end vs single-end, read length, depth, and strandedness) on downstream analysis

## How does sequencing work? 

Illumina sequencing is a commonly used technique within NGS techniques and relies on "sequencing by synthesis". This method determines the DNA sequence by detecting single nucleotides as they are added to a growing DNA strand. This method greatly reduces errors and prevents missed calls that often occur in strings of repeated nucleotides (homopolymers).



### 1. Library preparation for Illumina sequencing
   - The starting material depends upon the protocol being used. RNA-Seq uses RNA extracted from the sample set, which is then converted to cDNA. These molecules are long and initially incompatible with high-throughput sequencing techniques, including those from Illumina. This material is extracted and generally assessed for quality by a microvolume spectrophotometer, such as a NanoDrop.
  
     
### 2. The sample is fragmented, and the end tails are repaired
   - Illumina sequencing uses short fragments of DNA, usually 200 - 600 bp. In the case of RNA-Seq, RNA can be fragmented before or after cDNA synthesis.
   - Generally, fragmentation is performed directly on the RNA. Standard RNA-Seq kits will add divalent cations, such as magnesium or zinc, to a heated sample. This causes random hydrolysis of the phosphodiester bonds within the RNA's backbone.
   - CDNA fragmentation can occur by enzymatic digestion with endonucleases or by sonication/shearing.
   - In either case, fragments are then end-repaired, A-tailed at the 3' end, and ligated to adapters.
   -  **NOTE:** In single-cell RNA-Seq, RNA is usually captured in short tags (the 3' end of each transcript), and in long-read RNA-Seq, such as PacBio sequencing, there is no fragmentation of full-length cDNA.%
  
     
### 4. Adapter Ligation
   - Adapters are short, synthetic double-stranded DNA sequences that are enzymatically joined, or ligated, to both ends of fragments so that they can bind to the flow cell. These allow for the samples to be barcoded and ensure all reads start from the same known sequence.
   - Illumina adapters have three functional parts: p5 & p7 flow cell binding sites, primer binding sites, and index barcodes. Typically, a prepared fragment would look like this: `[P5] [Index 1] [Read 1 primer site] --- cDNA Fragment(100-600bp) --- [Read 2 primer site] [Index 2] [P7]`
   - **To read more about adapters, visit [Illumina's website](https://support-docs.illumina.com/SHARE/AdapterSequences/Content/SHARE/AdapterSeq/Overview.htm)**
  
     
### 5. Amplification and size selection
   - Libraries can be PCR amplified to ensure that each fragment has adapters on both ends(this can cause duplication bias within the results)
   - Fragments that are incorrect in size are removed with gel or beads
  
     
### 6. Flow cell binding and cluster generation
   - The flow cell is a glass slide coated with a lawn of short, single-stranded oligonucleotides, either p5 or p7. The DNA fragments with adapters can hybridize with these oligos, and fragments without adapters are washed away.
     
![image](https://upload.wikimedia.org/wikipedia/commons/6/65/Cluster_Generation.png)

   *Image: ["Cluster Generation.png"](https://commons.wikimedia.org/wiki/File:Cluster_Generation.png) by DMLapato, via Wikimedia Commons, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/), unchanged.*

   - Hybridization with the flow cell occurs at the complementary oligo and binds by base pairing. There are millions of single molecules randomly attached to the glass surface. These molecules will need multiple copies for their fluorescent signals to be accurately detected.
   - Bridge amplification is used to make these copies. These tethered fragments bound to oligos bend over and hybridize to a nearby oligo. DNA polymerase makes a new complementary strand, tethered to this new oligo from the opposite end. This double-stranded bridge is denatured, leaving two single-stranded copies attached at each oligo. This process is repeated, so dense clusters of identical fragments are formed.
   - After many dense clusters or identical nucleotides are formed and bound to this flow cell, sequencing by synthesis occurs. Each fragment within the cluster acts as a template strand while DNA polymerase synthesizes the complementary strand. The sequencer records the fluorescence of each added nucleotide within the cycle, averaging across the cluster to gain an accurate reading as to which nucleotide is added.
   - To read each base as it is added, the nucleotides are tagged with a fluorescent dye and a reversible terminator, meaning each cycle can only add one tagged nucleotide. An image is taken of the flow cell each cycle, and the incorporated base is recorded.
   - This is output as a FASTQ file, where the sequence is recorded and the accuracy of the color signal intensity is added as a quality score. These FASTQ files are the input to our sequencing analysis pipelines.


#### Illumina Sequencing is complicated, please watch this [video](https://www.youtube.com/watch?v=fCd6B5HRaZ8&pp=ygUdaWxsdW1pbmEgc2VxdWVuY2luZyBleHBsYWluZWQ%3D) for a visual explination on the process. 

#SHIFT + ENTER TO SEE VIDEO
<iframe width="560" height="315" src="https://www.youtube.com/embed/fCd6B5HRaZ8?si=OgezpcU_TKRf6ufF" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

## Impacts of Sequencing Choices on Downstream Results

1. **Read length:** Illumina generates short reads of 50-300bp within the FASTQ
   - Longer reads improve alignment, capture splice junctions better, and are much less cost-effective
   - To increase read length, you must use a different sequencing technique, like NanoPore
   - Generally, if you are investigating isoforms or splice junctions, then it is best to use longer reads, if you have the resources
2. **Single-End vs Paired-End Reads**
   - Single-end reads only sequence one end of the fragment. This is more affordable and optimal when the fragment structure is not crucial, such as for basic ChIP-Seq or simple gene expression profiling in RNA-Seq.
   - Paired-end reads sequence both ends of the fragment, providing information about fragment length and connectivity. This improves accuracy in repetitive regions and is needed when assessing structural variants, fusion transcripts, and isoform reconstructions.
3. **Strandedness**
   - preserves information on which strand of the cDNA the RNA came from. This can help determine if genes overlap on opposite strands and help to elucidate expression signals that can be ambiguous in the cases of this overlap.
   - In modern RNA-Seq, most libraries are stranded.
4. **Sequencing Depth** (Coverage)
   - Depth is how many times each base within a transcript is sequenced on average. In RNA-Seq, higher depth increases the detection of lowly expressed genes.
   - Typically, bulk RNA-Seq has 20 to 50 million reads per sample. In this case, a rare transcript may be measured several times over. If the sequencing depth was reduced to 5 million reads per sample, these rare transcripts may not be detected frequently enough to quantify them.
5. **Platform Choice**
   - Illumina is usually used in short-read, high-throughput sequencing. It offers coherent and cheap results for bulk RNA-Seq. For long-read RNA-Seq, such as single-cell, [PacBio's HiFi](https://www.pacb.com/technology/hifi-sequencing/) and [Oxford's Nanopore](https://nanoporetech.com/platform/technology) offer full isoform sequencing.
  
# How to choose the Sequencing Method
To choose the right sequencing method, experimental design must be considered. If you were to ask "Which isoforms are expressed within a certain tissue?", long-read sequencing would usually be the best choice. This does come at a higher cost and can be less accurate than Illumina sequencing. If you were to ask, "How does gene expression change across treatments?", then Illumina short-read sequencing would answer that question more accurately at a lower cost. 

**Each time you are designing an experiment, you should ask yourself:** 
- What is my main biological question?
- What resolution do I need to answer it?
- What can I afford?